# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
    ]

    yield from records

In [3]:
params = j2v.Hyperparameters(
    d_model=16,
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-18 20:17:46.216 | INFO     | json2vec.architecture.root:__init__:178 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(
    accelerator="mps",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    # enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/j

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9983429312705994, 0.9969947338104248],
│   │   │   'null': [0.0005860828096047044, 0.000657105294521898],
│   │   │   'padded': [0.0006993775605224073, 0.0017562373541295528],
│   │   │   'masked': [0.00014079399988986552, 0.00027344486443325877],
│   │   │   'other': [0.0002306671958649531, 0.00031823632889427245]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9929932355880737, 0.9882615208625793],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9929932355880737},
│   │   │   │   │   {'label': 'cool', 'probability': 0.007006636820733547}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9882615208625793},
│   │   │   │   │   {'label': 'warm', 'probability': 0.011738397181034088}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.05476762354373932,
│   │   │   │   0.0004737959534395486,
│   │   │   │   -0.1401662528514862,
│   │   │   │   -0.06852606683969498,
│   │   │   │   0.26019930839538574,
│   │   │   │   -0.03756164014339447,
│   │   │   │   -0.04186168685555458,
│   │   │   │   -0.016498634591698647,
│   │   │   │   0.6624855399131775,
│   │   │   │   0.24746173620224,
│   │   │   │   -0.30099624395370483,
│   │   │   │   -0.03913553059101105,
│   │   │   │   0.21118588745594025,
│   │   │   │   0.15712754428386688,
│   │   │   │   -0.33654409646987915,
│   │   │   │   -0.35596850514411926
│   │   │   ],
│   │   │   [
│   │   │   │   0.11047502607107162,
│   │   │   │   -0.12046199291944504,
│   │   │   │   0.18656162917613983,
│   │   │   │   -0.03935740888118744,
│   │   │   │   0.15048900246620178,
│   │   │   │   -0.327428936958313,
│   │   │   │   0.1120079755783081,
│   │   │   │   -0.3224726915359497,
│   │   │   │   -0.6711245179176331,
│   │   │   │   0.11613844335079193,
│   │   │   │   0.2810705304145813,
│   │   │   │   -0.17670311033725739,
│   │   │   │   0.16336968541145325,
│   │   │   │   0.15116174519062042,
│   │   │   │   0.2482820749282837,
│   │   │   │   0.0723491907119751
│   │   │   ]
│   │   ]
│   }
}

In [8]:
model.plot(detail=True)

╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ n_heads: 4                                                                                                           │
│ d_model: 16                                                                                                          │
│ target: ['record/label']                                                                                             │
│ embed: ['record']                                                                                                    │
│                                                                                                                      │
│   ┏━ record (array) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│   ┃ n_heads: 4                                                                                                     ┃ │
│   ┃ attention: mha                                                                                                 ┃ │
│   ┃ max_length: 1                                                                                                  ┃ │
│   ┃ n_outputs: 1                                                                                                   ┃ │
│   ┃ n_linear: 1                                                                                                    ┃ │
│   ┃ n_layers: 1                                                                                                    ┃ │
│   ┃                                                                                                                ┃ │
│   ┃   ╭─ color (category) ───────────────────────────────────────────────────────────────────────────────────────╮ ┃ │
│   ┃   │ address: record/color                                                                                    │ ┃ │
│   ┃   │ n_heads: 4                                                                                               │ ┃ │
│   ┃   │ query: [*].color                                                                                         │ ┃ │
│   ┃   │ pooling: query                                                                                           │ ┃ │
│   ┃   │ weight: 1.0                                                                                              │ ┃ │
│   ┃   │ n_linear: 1                                                                                              │ ┃ │
│   ┃   │ max_vocab_size: 16                                                                                       │ ┃ │
│   ┃   │ n_bands: 8                                                                                               │ ┃ │
│   ┃   │ p_unavailable: 0.01                                                                                      │ ┃ │
│   ┃   │ topk: []                                                                                                 │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃   │ state                                                                                                    │ ┃ │
│   ┃   │   vocabulary_size: 6                                                                                     │ ┃ │
│   ┃   │   vocabulary:                                                                                            │ ┃ │
│   ┃   │     ['green', 'red', 'purple', 'orange', 'yellow',                                                       │ ┃ │
│   ┃   │      'blue']                                                                                             │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃   │ counters.state                                                                                           │ ┃ │
│   ┃  

'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="UTF-8">\n<style>\n\nbody {\n    color: #000000;\n    background-color: #ffffff;\n}\n</style>\n</head>\n<body>\n    <pre style="font-family:Menlo,\'DejaVu Sans Mono\',consolas,\'Courier New\',monospace"><code style="font-family:inherit">╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮\n│ n_heads: 4                                                                                                           │\n│ d_model: 16                                                                                                          │\n│ target: [&#x27;record/label&#x27;]                                                                                             │\n│ embed: [&#x27;record&#x27;]                                                                                                    │\n│                                                                                    